# 1. What you'll learn

        - trace expectation and maximization conceptually
- inspect posterior membership uncertainty
- select component count with BIC

        **The one question this notebook answers:** How do soft assignments and covariance shapes improve on nearest-centroid clustering?

# 2. Setup

This lesson keeps gaussian mixture models small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Three overlapping anisotropic Gaussian components have known labels and covariances. Their overlap makes a probability vector more honest than a hard label.

**Small example:** If one point has component densities [0.06,0.03] and equal weights, responsibilities normalize to [2/3,1/3], not an absolute declaration.

In [ ]:
rng=np.random.default_rng(SEED);means=[[-3,0],[0,2.5],[3,0]];covs=[[[1.4,.9],[.9,1]],[[.5,-.2],[-.2,1.5]],[[1,-.7],[-.7,1.2]]];parts=[rng.multivariate_normal(m,c,260) for m,c in zip(means,covs)];X=np.vstack(parts);truth=np.repeat(range(3),260);print("Shape:",X.shape,"component counts:",pd.Series(truth).value_counts().to_dict())

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(X[:,0],X[:,1],s=10,color="#176b66");axes[0].set_title("Unlabelled overlap")
axes[1].scatter(X[:,0],X[:,1],c=truth,cmap="viridis",s=10);axes[1].set_title("Truth, evaluation only")
for k in range(3): sns.kdeplot(x=X[truth==k,0],y=X[truth==k,1],levels=4,ax=axes[2])
axes[2].set_title("Elliptical density contours");plt.tight_layout();plt.show()

**Takeaway:** The unlabeled cloud contains overlapping rather than separated islands.

**Takeaway:** Known truth reveals ambiguity near component intersections.

**Takeaway:** Full covariance ellipses express orientations that spherical distance cannot.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
Xs=StandardScaler().fit_transform(X);ks=range(1,7);models={k:GaussianMixture(n_components=k,covariance_type="full",n_init=5,random_state=SEED).fit(Xs) for k in ks};print("Scaled shape:",Xs.shape)

# 6. Train

        Training turns the assumptions behind gaussian mixture models into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Initialize component weights, means, and covariances.
2. E-step: compute each row's responsibility for every component.
3. M-step: update parameters using responsibilities as fractional counts.
4. Repeat until log likelihood improvement is negligible.

In [ ]:
bics=[models[k].bic(Xs) for k in ks];chosen=list(ks)[int(np.argmin(bics))];model=models[chosen];prob=model.predict_proba(Xs);labels=prob.argmax(1);print("BIC:",dict(zip(ks,np.round(bics,1))),"chosen:",chosen,"converged:",model.converged_)

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
ari=adjusted_rand_score(truth,labels);certainty=prob.max(1);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(list(ks),bics,marker="o");axes[0].set(title="BIC selects complexity",xlabel="components")
axes[1].scatter(X[:,0],X[:,1],c=labels,cmap="viridis",s=10);axes[1].set_title(f"Hard view · ARI {ari:.3f}")
p=axes[2].scatter(X[:,0],X[:,1],c=certainty,cmap="magma",s=12,vmin=.33,vmax=1);fig.colorbar(p,ax=axes[2]);axes[2].set_title("Maximum responsibility");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Random-label ARI baseline≈0")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Wrong covariance restrictions distort anisotropic groups, while too many components use Gaussians to memorize density details. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
bad=GaussianMixture(n_components=3,covariance_type="spherical",n_init=5,random_state=SEED).fit(Xs);over=GaussianMixture(n_components=8,covariance_type="full",n_init=5,random_state=SEED).fit(Xs)
print(f"Full-covariance ARI={ari:.3f}; forced-spherical ARI={adjusted_rand_score(truth,bad.predict(Xs)):.3f}");print(f"BIC chosen={model.bic(Xs):.1f}; overfit 8-component BIC={over.bic(Xs):.1f}")

**Use this when…** data plausibly arise from overlapping elliptical subpopulations and membership uncertainty matters.

        **Don't use this when…** clusters curve, tails are heavy, components are not generative groups, or high dimension makes covariance unstable.

        ## Try this

        1. Compare diagonal, tied, spherical, and full covariance BIC.
2. Plot a point's full responsibility vector.
3. Add outliers and watch Gaussian means/covariances move.